# Fractional differentiate

This notebook will cover exercise answer.

* Exercise 5.6

As we go along, there will be some explanations.

Stationarity is a key concept in time-series, by now the idea itself has been demostrated in previous notebooks (Feat Importance).

Most of the functions below can be found under research/Features and /Ensemble

Contact: boyboi86@gmail.com

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cqrlib as rs

%matplotlib inline

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import coint

p = print

#pls take note of version
#numba 0.49.1 #https://github.com/numba/numba/issues/4255
#numpy 1.17.3
#pandas 1.0.3
#sklearn 0.21.3

dollar = pd.read_csv('../sample-data/dollar_bars.txt', 
                 sep=',', 
                 header=0, 
                 parse_dates = True, 
                 index_col=['date_time'])

# For most part of the func we only use 'close'

close = dollar['close'].to_frame()

In [ ]:
ffd_series = close.apply(np.log).cumsum()
ffd_series = rs.fracDiff_FFD(ffd_series, 
                    d = 1.99999889, 
                    thres=1e-5
                   ).dropna()

adf_pval = adfuller(ffd_series.squeeze(), 
                    maxlag = 1,
                    regression='c', 
                    autolag=None)[1]

p("\nADF pVal: {0:.5f} with critical value: {1}%".format(adf_pval, 5)) #confirm stationarity

In [ ]:
ffd_series.head()

In [ ]:
ffd_series.tail()

In [ ]:
cs_event = rs.cs_filter(data = ffd_series, limit=(ffd_series.std() * 0.2))

p(len(cs_event))

**Note**

This is the real reason why cs_filter was created, it was meant to be applied to stationary series.

Our stationary series can act as our anchor for our mean-reversion strategy, that is why we are using cs_filter on this series to sample feature matrix.

However, due to the lack of data sample/ resource, I cannot use 2 x std. (In fact, we would have been able to increase our F1 score in previous chapters, if we had enough data samples.)

Based on my previous trials, when 1 x std is applied, the series itself will reduce heteroscedasticity even if it was not stationary.

If you have the means to sample 5 - 10 years data, feel free to change to 2 x std.

In [ ]:
#This portion requires combining events based on ffd_series with original dollar close using filtered index
# since filter is based on ffd, therefore index matrix must also be based on filtered ffd
df_mtx = pd.DataFrame(index = cs_event).assign(close = close,
                                                ffd_series = ffd_series).drop_duplicates().dropna()
df_mtx

In [ ]:
df_mtx['volatility'] = rs.vol(df_mtx.close, span0 = 50) #one of our features, since we do not have a side

df_mtx.dropna(inplace = True)

In [ ]:
vb = rs.vert_barrier(data = df_mtx.close, events = cs_event, period = 'days', freq = 5)

In [ ]:
# triple barrier events based on filter while data is also based on filtered index
tb = rs.tri_barrier(data = df_mtx.close, 
                    events = cs_event, 
                    trgt = df_mtx['volatility'], 
                    min_req = 0.0002, 
                    num_threads = 3, 
                    ptSl= [2,2], #2x barriers
                    t1 = vb, 
                    side = None)
    


In [ ]:
tb

In [ ]:
mlabel = rs.meta_label(data = df_mtx.close, 
                       events = tb, 
                       drop = 0.05) # because we do not have side, we need to drop rare labels

In [ ]:
mlabel['bin'].value_counts() #834

In [ ]:

troo = rs.mp_idx_matrix(data = df_mtx.close, events = tb)
#np.arange(troo.shape[1])
troo

**Quick Reference Guide/ Summary to func usage**

1. Apply cs_filter using stationary series (FFD)

2. Apply bband_as_side using close price series

3. Create a new dataframe, using both outputs with index = cs_filter output.

4. Apply func tri_barrier, Vert_barrier using new dataframe, before applying meta_label func.

5. Feed these meta lables and features to random forest.

Always include indicators, stationary series, volatility as part of your feature matrix if possible, since these are the features on how you derive meta-labels. 

As seen in previous exercises, ML models are smart enough to know what was the key parameters which meta-labels was labelled.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble.bagging import BaggingClassifier

base_estimate = DecisionTreeClassifier()

# common parameters to use, this ensemble method mimics monte-carlos as it randomly selects samples to form "forest"
n_estimate = 10
random_state = 42
max_samples = 35

In [ ]:
X = df_mtx.reindex(mlabel.index)
y = mlabel['bin']

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, shuffle=True, stratify=None)

X_train #quick check

In [ ]:
#warning, this is a heavy algo, will take up to 15 mins to complete with current parameter

rs_bag_cls=rs.BaggingClassifier(data = df_mtx.close,
                                events = tb,
                                base_estimator = base_estimate,
                                n_estimators = n_estimate, # the number of trees as a estimate
                                max_samples = max_samples, #if you put max_sample = 1.0, pls mentally get ready to wait
                                max_features = 1.0,
                                bootstrap = True, #must always be True
                                bootstrap_features=False,
                                oob_score = True, # You will need to have bootstrap = True
                                warm_start = False,
                                n_jobs = 1, # depending your machine. might become slower if you use all processors
                                random_state = random_state, #seed figure
                                verbose = 0)



# Fit and score
rs_bag_cls.fit(X_train, y_train)

In [ ]:
p("Out-of-bag Score: {0:.6f}".format(rs_bag_cls.oob_score_))

In [ ]:
y_pred = rs_bag_cls.predict(X_test)
y_prob = rs_bag_cls.predict_proba(X_test)[:,1] #we only want true positive probability

rs.report_matrix(y_test, y_pred, y_prob)

In [ ]:
skl_bag_cls = BaggingClassifier(base_estimator = base_estimate,
                                n_estimators = n_estimate, # the number of trees as a estimate
                                max_samples = max_samples, #if you put max_sample = 1.0, pls mentally get ready to wait
                                max_features = 1.0,
                                bootstrap = True, #must always be True
                                bootstrap_features=False,
                                oob_score = True, # You will need to have bootstrap = True
                                warm_start = False,
                                n_jobs = -1, # For parallel computering, -1 means use all
                                random_state = random_state, #seed figure
                                verbose = 0)

# Fit and score
skl_bag_cls.fit(X_train, y_train)

In [ ]:
y_pred1 = skl_bag_cls.predict(X_test)
y_prob1 = skl_bag_cls.predict_proba(X_test)[:,1]

rs.report_matrix(y_test, y_pred1, y_prob1)

In [ ]:
p("Out-of-bag Score: {0:.6f}".format(skl_bag_cls.oob_score_))

### Out-of-Bag score

* Original sklearn bagging method: Out-of-bag Score: 0.662093

* Sequential bootstrap bagging method: Out-of-bag Score: 0.655232

Comparing the original sklearn bagging ensemble using decision tree classifier, our OOB score is lower. 

This is correct, since the original algo from sklearn will always inflate their OOB score leading to a more bias outcome.

If you are not sure why, kindly refer to [AFML 4.1](https://github.com/boyboi86/AFML/blob/master/AFML%204.1.ipynb).

There was a comparison to why Kfold accuracy was lower compared to OOB (due to sklearn bootstrap method).